In [8]:
import pandas as pd
import logging
import requests
import json
import os
import http.client

In [14]:
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type

retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms 
    retry=retry_if_exception_type([requests.ConnectionError])
)()

In [16]:
# Set up logging
logging.basicConfig(filename='DOI.log', filemode='w', format='%(asctime)s - %(levelname)s - %(message)s', level=logging.INFO)

@retry_on_communication_error
def get_publisher_ids(csv_file):
    """
    Read DOIs from a CSV file, query the CrossRef API, and return a new DataFrame with publisher IDs.
    """
    logging.debug("Entering get_publisher_ids function")
    logging.info(f"Reading DOIs from CSV file: {csv_file}")

    # Read DOIs from CSV file
    df = pd.read_csv(csv_file)

    # Extract DOIs from 2nd column of the csv file
    dois = df.iloc[:, 1].tolist() 

    publisher_ids = []

    # Iterate over the list of DOIs
    for doi in dois:

        # Define the base URL with DOI and select parameters
        url = f"https://api.crossref.org/works?filter=doi:{doi}&select=publisher"

        # Make GET request to the CrossRef API
        response = requests.get(url)

        # Check for successful response (status code 200)
        if response.status_code == 200:

            # Parse JSON response
            data = json.loads(response.text)
            if "message" in data:
                if "items" in data["message"] and data["message"]["items"]:
                    if len(data["message"]["items"]) > 0:
                        first_item = data["message"]["items"][0]
                        if isinstance(first_item, dict):  # Check if first_item is a dictionary
                            for key, value in first_item.items():
                                publisher_ids.append(value)  # Append the value to the list
                            else:
                                publisher_ids.append(str(first_item))  # Convert the list to a string and append it
                    else:
                        # Handle failed API request
                        logging.error(f"Error: API request failed {response.status_code} for {doi}")
                        publisher_ids.append(None)  # Append None if the API request fails
    # Create a new DataFrame with the original DataFrame adding publisher_ids list
    new_df = pd.DataFrame(list(zip(df.values.tolist(), publisher_ids)), columns=['PMID', 'doi', 'publisher'])

    logging.info("New DataFrame created:")
    logging.info(new_df)

    logging.debug("Exiting get_publisher_ids function")
    return new_df

# Example usage
csv_file_path = "PMID_doi_cleaned.csv"
new_df = get_publisher_ids(csv_file_path)

# Export the new_df to a new CSV file called PMID_Publisher.csv
new_df.to_csv('DOI_Publisher.csv', index=False)
